# Random search de hiperparâmetros - Sprint 4

Define os espaços de busca de Random Forest, Extra Trees, Gradient Boosting e Ridge e os percorre por amostragem aleatória sob validação cruzada leave-one-group-out no conjunto de treino, na mesma formulação da solução documentada na Seção 4.3.

A Parte I monta a infraestrutura e declara as distribuições e o orçamento de sorteios. A Parte II confere a reprodução da Seção 4.3 e executa a busca em uma área, como teste de integridade, estimando o tempo da execução completa. A Parte III, desligada aqui, executa a busca nas sete áreas e é a entrega do card seguinte.

**Regras de método.** A partição é lida de arquivo. O conjunto de teste não é lido. Toda estatística aprendida dos dados, inclusive o patamar e a escala do componente e o escalonamento do Ridge, é ajustada dentro da dobra. A avaliação incide apenas sobre valores medidos. As métricas são calculadas uma vez sobre o conjunto das previsões fora da dobra, e não como média entre dobras, que é o mesmo cálculo do notebook de modelagem.

## Parte I - Infraestrutura

### 1. Configuração

Concentra os parâmetros fixos. `TRATAMENTO` é mantido em máscara porque é a condição que reproduz a solução documentada: no formato longo, preencher o ausente contamina o patamar e a escala do componente e piora o resultado, sobretudo no Coil Temps.

As duas sementes são distintas e têm papéis diferentes. `SEMENTE_BUSCA` governa o sorteio das combinações e é o que torna a busca reproduzível; `SEMENTE_MODELO` governa o *bootstrap* e a subamostragem dentro de cada modelo. `N_SORTEIOS` é o orçamento por par de área e algoritmo, igual para os quatro algoritmos, de modo que a comparação entre eles não seja contaminada por orçamentos diferentes.

`LIMIAR_P` é o nível de significância do teste que decide se a melhor combinação sorteada supera a configuração atual.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json, os, time, warnings

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform, wilcoxon
from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEMENTE_MODELO = 42
SEMENTE_BUSCA = 7
N_SORTEIOS = 30
VERSAO = "perfil"
TRATAMENTO = "mascara"
LIMIAR_P = 0.05
AREAS = ["oven_hmi", "ct_knobs", "ipc_cooling", "main_vent_sys",
         "mainback", "ipc_comp", "coil_temps"]
PISO = {"oven_hmi": 0.58, "ct_knobs": 0.92, "ipc_cooling": 2.15,
        "main_vent_sys": 1.16, "mainback": 1.33, "ipc_comp": 3.30,
        "coil_temps": 6.36}
PREFIXOS_FEATURES = ("blower_direction_", "housing_", "board_position_", "run_case_",
                     "losses_", "deflector_hole_", "material_coil_support_", "cfg_", "reason_")
# Colunas que existem na base mas nao entram como entrada nem como alvo.
EXCLUIDAS = ("loss_profile_order_",)
PASTA_PROJETO = None
PASTA_SAIDA = os.path.join(PASTA_PROJETO or ".", "resultados_random")
os.makedirs(PASTA_SAIDA, exist_ok=True)

# Os CSVs sao lidos apenas do colabs_S4: a pasta do caderno, o Colab ou o colabs_S4
# a partir da raiz do repositorio.
BASES = [".", "/content", os.path.join("notebooks", "colabs_S4")]


def resolve(arquivo):
    '''Localiza um arquivo do projeto pelas pastas usuais do repositorio.'''
    for base in ([PASTA_PROJETO] if PASTA_PROJETO else []) + BASES:
        for sub in ("",):
            caminho = os.path.join(base, sub, arquivo)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(f"{arquivo} nao encontrado a partir de {os.path.abspath('.')}. "
                            "Preencha PASTA_PROJETO.")


print("Diretorio:", os.path.abspath("."))
print(f"versao: {VERSAO} | tratamento: {TRATAMENTO} | orcamento: {N_SORTEIOS} sorteios por par")
print(f"sementes: busca={SEMENTE_BUSCA}, modelo={SEMENTE_MODELO}")

### 2. Partição

Lida de arquivo, nunca sorteada. Os ensaios de teste recebem dobra ausente e são descartados na montagem do formato longo, antes de qualquer outra operação, de modo que nenhuma célula posterior tem acesso a eles.

In [ ]:
particao = pd.read_csv(resolve("particao_treino_teste.csv")).sort_values("run_idx")
DOBRA = dict(zip(particao["run_idx"].astype(int), particao["dobra"]))
RUN_TESTE = set(particao.loc[particao["conjunto"] == "teste", "run_idx"].astype(int))
DOBRAS = sorted(d for d in set(DOBRA.values()) if pd.notna(d))

assert len(DOBRAS) == 17, "a particao deveria ter 17 dobras de treino"
print(f"{len(DOBRA) - len(RUN_TESTE)} ensaios de treino em {len(DOBRAS)} dobras | "
      f"{len(RUN_TESTE)} ensaios de teste reservados e nao lidos")

### 3. Formato longo

Cada área vira uma tabela com uma linha por par de ensaio e componente. A ordem das linhas dos datasets tratados é a ordem de `run_idx`, que é o que permite associar cada linha à partição.

`carrega_area` falha em três situações: dataset desalinhado com a partição, base sem os atributos do motivo do ensaio (versão anterior dos arquivos) e ensaio de teste presente no resultado. Linhas sem medição são descartadas aqui, o que no tratamento por máscara equivale a excluí-las do ajuste e da avaliação.

In [ ]:
def carrega_area(area):
    '''Devolve (longo, colunas_de_configuracao) apenas com ensaios de treino medidos.'''
    df = pd.read_csv(resolve(f"dataset_{area}_{VERSAO}.csv"))
    assert len(df) == len(DOBRA), f"{area}: dataset e particao divergem"
    df["run_idx"] = np.arange(len(df))
    X = [c for c in df.columns if c.startswith(PREFIXOS_FEATURES)]
    assert any(c.startswith(("cfg_", "reason_")) for c in X), (
        f"{area}: base sem os atributos do motivo; regenere com preparacao_datasets")
    alvos = [c for c in df.columns if c not in X + ["run_idx"] and not c.startswith(EXCLUIDAS)]
    longo = df.melt(id_vars=["run_idx"] + X, value_vars=alvos,
                    var_name="componente", value_name="y")
    longo["dobra"] = longo["run_idx"].map(DOBRA)
    longo = longo[longo["dobra"].notna() & longo["y"].notna()].reset_index(drop=True)
    assert not set(longo["run_idx"]) & RUN_TESTE, f"{area}: ensaio de teste lido"
    return longo, X


def verifica_dobras(longo, X):
    '''Nenhuma configuracao pode estar no treino e na validacao da mesma dobra.'''
    chave = longo[X].astype(str).agg("|".join, axis=1)
    for d in DOBRAS:
        comum = set(chave[longo["dobra"] != d]) & set(chave[longo["dobra"] == d])
        assert not comum, f"dobra {d}: configuracao no treino e na validacao"

### 4. Validação cruzada com o patamar e a escala dentro da dobra

O patamar (média) e a escala (desvio-padrão) de cada componente são calculados apenas sobre as linhas de treino da dobra e acrescentados ao vetor de configuração, como na Seção 4.3. Por isso a validação é um laço próprio, e não o `RandomizedSearchCV` do scikit-learn: o `RandomizedSearchCV` não recalcula esses atributos por dobra e promedia o erro entre dobras de tamanhos diferentes, o que não coincide com o cálculo adotado pelo projeto. O `ParameterSampler`, que é a parte de amostragem do `RandomizedSearchCV`, é reaproveitado.

O Ridge é instanciado como pipeline de escalonamento e regressão, e o pipeline é ajustado dentro do laço, de modo que o escalonador aprenda apenas com as linhas de treino de cada dobra.

`avalia` devolve as métricas agregadas e o erro médio de cada dobra, que alimenta o teste pareado da seção 7.

In [ ]:
def avalia(longo, X, fabrica):
    '''Leave-one-group-out. Metricas sobre o conjunto das previsoes fora da dobra.'''
    features = X + ["patamar", "escala"]
    reais, preditos, mae_dobra = [], [], {}
    for d in DOBRAS:
        tr = longo[longo["dobra"] != d]
        va = longo[longo["dobra"] == d]
        if va.empty:
            continue
        stats = tr.groupby("componente")["y"].agg(patamar="mean", escala="std")
        A = tr.join(stats, on="componente")
        B = va.join(stats, on="componente")
        modelo = fabrica().fit(A[features], A["y"])
        p = modelo.predict(B[features])
        reais.append(B["y"].to_numpy())
        preditos.append(p)
        mae_dobra[int(d)] = float(np.mean(np.abs(B["y"].to_numpy() - p)))
    r, p = np.concatenate(reais), np.concatenate(preditos)
    return {"MAE": float(mean_absolute_error(r, p)),
            "RMSE": float(np.sqrt(mean_squared_error(r, p))),
            "R2": float(r2_score(r, p)),
            "vies": float(np.mean(p - r)),
            "n_previsoes": int(len(r))}, mae_dobra

### 5. Espaços de busca

Cada parâmetro recebe uma distribuição, e não uma lista de valores. Os suportes foram definidos pelo tamanho do treino no formato longo: uma dobra tem entre cerca de 170 linhas (Oven HMI) e 1.500 linhas (IPC Comp), com aproximadamente 30 colunas de entrada, de modo que os valores herdados da formulação por componente, ajustada para cerca de 40 linhas, não servem de centro do espaço.

**Random Forest e Extra Trees.** O número de árvores fica fixo em 300: em *bagging*, mais árvores não aumentam o sobreajuste, apenas estabilizam a média, e sorteá-lo gastaria orçamento sem responder a pergunta alguma. Variam os três parâmetros que controlam a complexidade de cada árvore.

| Parâmetro | Distribuição | Razão |
|---|---|---|
| `max_features` | uniforme em [0,4; 1,0] | 1,0 é o padrão de regressão e a configuração atual; a raiz do total foi medida e custa 1,55 °C (Seção 4.3), por isso o suporte não desce abaixo de 40% das colunas |
| `max_depth` | inteiro uniforme em [4; 30] | com até 1.500 linhas, uma árvore de profundidade 30 é indistinguível de uma sem limite, que é o caso da configuração atual |
| `min_samples_leaf` | inteiro uniforme em [1; 15] | 1 é o padrão; folhas maiores suavizam a previsão em áreas de alvo ruidoso, como Coil Temps e IPC Comp |

**Gradient Boosting.** A configuração atual (300 árvores, taxa de 0,05, profundidade 2, folha mínima 2, subamostragem de 0,8) foi fixada para a formulação por componente. No formato longo, profundidade 2 tende a ser insuficiente, porque o modelo precisa combinar o patamar do componente com a configuração do ensaio, e por isso o suporte da profundidade vai além dela. Como a busca aleatória não paga custo combinatório por dimensão adicional, a subamostragem também é sorteada, o que não era viável na grade.

| Parâmetro | Distribuição | Razão |
|---|---|---|
| `learning_rate` | log-uniforme em [0,01; 0,3] | escala logarítmica porque o efeito da taxa é multiplicativo; o suporte cobre o atual 0,05 com uma ordem de grandeza para cada lado |
| `n_estimators` | inteiro uniforme em [150; 800] | sorteado junto com a taxa, que é o par que decide o ponto de parada |
| `max_depth` | inteiro uniforme em [2; 6] | do atual até a profundidade que capta a interação entre componente e configuração |
| `min_samples_leaf` | inteiro uniforme em [2; 30] | 2 é o atual; 20 é o padrão do HistGradientBoosting, adequado à formulação por área (registro da Sprint 3) |
| `subsample` | uniforme em [0,6; 1,0] | 0,8 é o atual; o extremo 1,0 recupera o *boosting* determinístico |

**Ridge.** Varia apenas a regularização, log-uniforme de 10⁻³ a 10³, faixa em que a busca aleatória é estritamente melhor que a grade: o parâmetro é contínuo e é o único da dimensão, de modo que os 30 sorteios cobrem 30 valores distintos, contra os 13 pontos tabelados de uma grade logarítmica. O escalonamento é aplicado a todas as colunas, inclusive às binárias, para que a penalidade atue de forma equivalente sobre o patamar, a escala e os indicadores.

A configuração atual de cada algoritmo **não** é sorteada: ela é avaliada à parte, sob o mesmo protocolo, e serve de referência para medir o ganho do tuning.

In [ ]:
ESPACOS = {
    "random_forest": {
        "max_features": uniform(0.4, 0.6),        # [0,4; 1,0]
        "max_depth": randint(4, 31),
        "min_samples_leaf": randint(1, 16),
    },
    "extra_trees": {
        "max_features": uniform(0.4, 0.6),
        "max_depth": randint(4, 31),
        "min_samples_leaf": randint(1, 16),
    },
    "gradient_boosting": {
        "learning_rate": loguniform(0.01, 0.3),
        "n_estimators": randint(150, 801),
        "max_depth": randint(2, 7),
        "min_samples_leaf": randint(2, 31),
        "subsample": uniform(0.6, 0.4),           # [0,6; 1,0]
    },
    "ridge": {
        "alpha": loguniform(1e-3, 1e3),
    },
}

FIXOS = {
    "random_forest": {"n_estimators": 300, "random_state": SEMENTE_MODELO, "n_jobs": -1},
    "extra_trees": {"n_estimators": 300, "random_state": SEMENTE_MODELO, "n_jobs": -1},
    "gradient_boosting": {"random_state": SEMENTE_MODELO},
    "ridge": {},
}

ATUAL = {
    "random_forest": {"max_features": 1.0, "max_depth": None, "min_samples_leaf": 1},
    "extra_trees": {"max_features": 1.0, "max_depth": None, "min_samples_leaf": 1},
    "gradient_boosting": {"learning_rate": 0.05, "n_estimators": 300, "max_depth": 2,
                          "min_samples_leaf": 2, "subsample": 0.8},
    "ridge": {"alpha": 1.0},
}

# Suporte numerico de cada distribuicao, usado para sinalizar vencedoras no extremo.
SUPORTE = {
    "random_forest": {"max_features": (0.4, 1.0), "max_depth": (4, 30),
                      "min_samples_leaf": (1, 15)},
    "extra_trees": {"max_features": (0.4, 1.0), "max_depth": (4, 30),
                    "min_samples_leaf": (1, 15)},
    "gradient_boosting": {"learning_rate": (0.01, 0.3), "n_estimators": (150, 800),
                          "max_depth": (2, 6), "min_samples_leaf": (2, 30),
                          "subsample": (0.6, 1.0)},
    "ridge": {"alpha": (1e-3, 1e3)},
}


def fabrica(algoritmo, params):
    '''Devolve uma funcao que instancia o modelo com os parametros dados.'''
    kw = {**FIXOS[algoritmo], **params}
    if algoritmo == "random_forest":
        return lambda: RandomForestRegressor(**kw)
    if algoritmo == "extra_trees":
        return lambda: ExtraTreesRegressor(**kw)
    if algoritmo == "gradient_boosting":
        return lambda: GradientBoostingRegressor(**kw)
    if algoritmo == "ridge":
        return lambda: make_pipeline(StandardScaler(), Ridge(**kw))
    raise ValueError(algoritmo)


def sorteia(algoritmo, n=N_SORTEIOS, semente=SEMENTE_BUSCA):
    '''Combinacoes amostradas. Mesma semente e mesmo orcamento: mesma lista.'''
    amostras = list(ParameterSampler(ESPACOS[algoritmo], n_iter=n, random_state=semente))
    return [{k: (float(v) if isinstance(v, (float, np.floating)) else int(v))
             for k, v in a.items()} for a in amostras]


for alg in ESPACOS:
    a = sorteia(alg)
    assert a == sorteia(alg), f"{alg}: amostragem nao reproduzivel"
    print(f"{alg:18s} {len(a)} sorteios | {len(ESPACOS[alg])} dimensoes | "
          f"1o sorteio: { {k: round(v, 4) if isinstance(v, float) else v for k, v in a[0].items()} }")

### 6. Execução de uma busca e escolha da configuração

`roda_busca` avalia a configuração atual e as `N_SORTEIOS` combinações amostradas de um algoritmo em uma área, na ordem do sorteio, e grava o resultado completo em `resultados_random/<area>__<algoritmo>.csv`. A ordem é preservada porque é ela que permite traçar a curva de melhor MAE contra o número de sorteios, que é a evidência de que o orçamento saturou. Se o arquivo já existir, é lido em vez de recalculado, o que permite interromper e retomar a execução das sete áreas.

In [ ]:
def roda_busca(area, algoritmo, refazer=False, verbose=True):
    destino = os.path.join(PASTA_SAIDA, f"{area}__{algoritmo}.csv")
    if os.path.exists(destino) and not refazer:
        return pd.read_csv(destino)
    longo, X = carrega_area(area)
    verifica_dobras(longo, X)
    candidatos = [(-1, ATUAL[algoritmo])] + list(enumerate(sorteia(algoritmo)))
    linhas, t0 = [], time.perf_counter()
    for ordem, params in candidatos:
        t = time.perf_counter()
        met, por_dobra = avalia(longo, X, fabrica(algoritmo, params))
        linhas.append({"area": area, "algoritmo": algoritmo, "ordem": ordem,
                       "atual": ordem == -1, "params": json.dumps(params, sort_keys=True),
                       **met, "mae_dobras": json.dumps(por_dobra),
                       "tempo_s": round(time.perf_counter() - t, 2)})
    r = pd.DataFrame(linhas)
    r.to_csv(destino, index=False)
    if verbose:
        print(f"{area} | {algoritmo}: {len(r) - 1} sorteios em "
              f"{time.perf_counter() - t0:.0f} s | melhor MAE "
              f"{r.loc[~r['atual'], 'MAE'].min():.2f} °C | atual "
              f"{r.loc[r['atual'], 'MAE'].iloc[0]:.2f} °C")
    return r

### 7. Seleção, extremos do suporte e saturação do orçamento

`escolhe` compara a melhor combinação sorteada com a configuração atual por teste de Wilcoxon pareado sobre o erro médio das 17 dobras. A dobra é a unidade do teste, e não a previsão, porque as previsões de uma mesma configuração não são independentes entre si. A melhor combinação só substitui a atual quando a diferença é significativa; caso contrário, o resultado é registrado como empate e a configuração atual é mantida, por ser a mais simples de justificar.

Duas verificações são próprias da busca aleatória e não teriam sentido em uma grade:

`no_extremo` sinaliza os parâmetros cuja vencedora caiu nos 10% externos do suporte sorteável. Como a amostragem é contínua, o extremo não é um valor tabelado, e sim a fronteira do suporte: uma vencedora ali indica que o ótimo pode estar fora dele. Extremos que não admitem extensão, como usar todas as colunas ou uma folha de um único registro, não são sinalizados.

`saturacao` devolve a curva do melhor MAE acumulado ao longo dos sorteios e o índice do sorteio que produziu a vencedora. Se a vencedora aparece nos primeiros sorteios e a curva fica plana até o fim, o orçamento foi suficiente; se a melhora ocorre no fim do orçamento, ele é curto.

In [ ]:
def no_extremo(algoritmo, params, margem=0.10):
    '''Parametros cuja vencedora caiu nos 10% externos do suporte sorteavel.'''
    naturais = {"max_features": {"sup"}, "min_samples_leaf": {"inf"},
                "subsample": {"sup"}}      # extremos que nao admitem extensao
    log = {"learning_rate", "alpha"}       # amostrados em escala logaritmica
    fora = []
    for k, (lo, hi) in SUPORTE[algoritmo].items():
        if k not in params:
            continue
        v = params[k]
        if k in log:
            lo, hi, v = np.log10(lo), np.log10(hi), np.log10(v)
        faixa = hi - lo
        if v <= lo + margem * faixa and "inf" not in naturais.get(k, set()):
            fora.append(f"{k} (inferior)")
        elif v >= hi - margem * faixa and "sup" not in naturais.get(k, set()):
            fora.append(f"{k} (superior)")
    return fora

def saturacao(r):
    '''Curva de melhor MAE acumulado sobre os sorteios, na ordem da amostragem.'''
    s = r[~r["atual"]].sort_values("ordem")
    curva = s["MAE"].cummin().to_numpy()
    return curva, int(np.argmin(curva)) + 1


def escolhe(r):
    '''Compara a melhor combinacao sorteada com a atual e decide qual manter.'''
    s = r[~r["atual"]]
    melhor = s.loc[s["MAE"].idxmin()]
    atual = r[r["atual"]].iloc[0]
    alg = melhor["algoritmo"]
    a = json.loads(atual["mae_dobras"])
    b = json.loads(melhor["mae_dobras"])
    dobras = sorted(a)
    difs = np.array([a[d] - b[d] for d in dobras])      # ganho por dobra, atual - melhor
    p = float(wilcoxon([a[d] for d in dobras], [b[d] for d in dobras]).pvalue)
    ganho = atual["MAE"] - melhor["MAE"]
    piso = PISO[melhor["area"]]
    significante = p < LIMIAR_P and ganho > 0
    relevante = ganho >= piso
    troca = significante and relevante
    motivo = "-" if troca else ", ".join(
        m for m, ok in (("p >= 0,05", significante), ("ganho < piso", relevante)) if not ok)
    escolhida = melhor if troca else atual
    curva, indice = saturacao(r)
    return {"area": melhor["area"], "algoritmo": alg,
            "params_escolhidos": escolhida["params"],
            "decisao": "melhor sorteio" if troca else "empate: mantida a atual",
            "motivo_empate": motivo,
            "piso": piso,
            "MAE_atual": round(atual["MAE"], 3), "RMSE_atual": round(atual["RMSE"], 3),
            "R2_atual": round(atual["R2"], 3),
            "MAE_melhor": round(melhor["MAE"], 3), "RMSE_melhor": round(melhor["RMSE"], 3),
            "R2_melhor": round(melhor["R2"], 3),
            "delta_MAE": round(melhor["MAE"] - atual["MAE"], 3),
            "p_wilcoxon": round(p, 4),
            "passa_significancia": bool(significante),
            "delta_ep": round(float(difs.std(ddof=1) / np.sqrt(len(difs))), 3),
            "MAE": round(escolhida["MAE"], 3), "RMSE": round(escolhida["RMSE"], 3),
            "R2": round(escolhida["R2"], 3), "vies": round(escolhida["vies"], 3),
            "sorteio_vencedor": f"{indice}/{len(curva)}",
            "extremo": ", ".join(no_extremo(alg, json.loads(melhor["params"]))) or "-"}

## Parte II - Teste de integridade e estimativa de tempo

### 8. Configuração atual

Antes da busca, o Random Forest na configuração atual é rodado nas sete áreas. Os valores devem coincidir com a tabela da Seção 4.3 a menos de arredondamento, e o total de previsões deve ser 3.277. Divergência aqui indica que a base, a partição ou a formulação não são as da solução documentada, e invalida a busca.

In [ ]:
REFERENCIA_43 = {"oven_hmi": 2.42, "ct_knobs": 3.16, "ipc_cooling": 4.04,
                 "main_vent_sys": 4.68, "mainback": 5.68, "ipc_comp": 7.30,
                 "coil_temps": 11.71}
linhas, n_total, soma, TEMPO_POR_AREA = [], 0, 0.0, {}
for area in AREAS:
    t = time.perf_counter()
    longo, X = carrega_area(area)
    verifica_dobras(longo, X)
    met, _ = avalia(longo, X, fabrica("random_forest", ATUAL["random_forest"]))
    TEMPO_POR_AREA[area] = time.perf_counter() - t
    n_total += met["n_previsoes"]
    soma += met["MAE"] * met["n_previsoes"]
    linhas.append({"area": area, "linhas_validacao": met["n_previsoes"],
                   "MAE": round(met["MAE"], 2), "MAE_4.3": REFERENCIA_43[area],
                   "diferenca": round(met["MAE"] - REFERENCIA_43[area], 2),
                   "tempo_s": round(TEMPO_POR_AREA[area], 1)})
conferencia = pd.DataFrame(linhas)
print(conferencia.to_string(index=False))
print(f"\nAgregado: {soma / n_total:.2f} °C sobre {n_total} previsoes "
      "(Secao 4.3: 6,57 °C, 3.277)")
assert n_total == 3277, "numero de previsoes diverge da Secao 4.3"
assert conferencia["diferenca"].abs().max() < 0.25, "MAE diverge da Secao 4.3"

### 9. Busca completa - teste

Os quatro espaços são percorridos no CT Knobs, área pequena e de tempo curto, para verificar que todas as combinações sorteadas rodam sem erro e que a seleção funciona antes da execução nas sete áreas. O resultado desta seção é um teste, não a entrega do card de execução.

In [ ]:
AREA_TESTE = "ct_knobs"
teste = {alg: roda_busca(AREA_TESTE, alg) for alg in ESPACOS}
escolha_teste = pd.DataFrame([escolhe(r) for r in teste.values()])
print()
print(escolha_teste.drop(columns=["area", "params_escolhidos"]).to_string(index=False))

In [ ]:
print("Curva de melhor MAE acumulado por sorteio (°C):\n")
for alg, r in teste.items():
    curva, indice = saturacao(r)
    marcos = [1, 5, 10, 15, 20, 25, len(curva)]
    print(f"{alg:18s} " + "  ".join(f"n={m}: {curva[m - 1]:.2f}" for m in marcos) +
          f"   | vencedora no sorteio {indice}")

### 10. Estimativa de tempo da execução completa

O tempo de cada ajuste cresce com o número de linhas da área. A estimativa escala o tempo medido no CT Knobs pela razão entre o tempo do Random Forest atual em cada área e no CT Knobs, medido na seção 8. É uma aproximação: a razão vale para os modelos de árvore e superestima o Ridge, cujo custo é desprezível.

In [ ]:
tempo_teste = {alg: r["tempo_s"].sum() for alg, r in teste.items()}
fator = {a: TEMPO_POR_AREA[a] / TEMPO_POR_AREA[AREA_TESTE] for a in AREAS}
estimativa = pd.DataFrame({alg: {a: tempo_teste[alg] * fator[a] / 60 for a in AREAS}
                           for alg in ESPACOS}).round(1)
estimativa["total_min"] = estimativa.sum(axis=1).round(1)
print("Minutos estimados por area e algoritmo:")
print(estimativa.to_string())
print(f"\nTotal estimado: {estimativa['total_min'].sum():.0f} min em {os.cpu_count()} nucleo(s), "
      f"para {len(AREAS) * len(ESPACOS)} pares e {N_SORTEIOS} sorteios cada. "
      "Random Forest e Extra Trees usam todos os nucleos disponiveis.")

## Parte III - Execução nas sete áreas

Percorre os quatro espaços nas sete áreas, com o orçamento integral de `N_SORTEIOS` em todos os 28 pares. Coil Temps e IPC Comp usam o mesmo espaço e o mesmo orçamento das demais, sem tratamento especial, de modo que a comparação entre áreas não fica condicionada ao esforço de busca.

### 11. Execução dos 28 pares

Cada par grava o resultado completo de todos os sorteios em `resultados_random/<area>__<algoritmo>.csv`, e não apenas a vencedora: é esse arquivo que sustenta a curva de saturação e permite reexaminar a escolha sem repetir a busca. Pares já gravados são reaproveitados, o que torna a célula retomável depois de uma interrupção.

In [ ]:
t0 = time.perf_counter()
for area in AREAS:
    for alg in ESPACOS:
        roda_busca(area, alg)
print(f"\n28 pares concluidos em {(time.perf_counter() - t0) / 60:.1f} min")

### 12. Auditoria da execução

Antes de consolidar, confere-se que os 28 arquivos existem, que cada um traz o orçamento integral mais a configuração atual, e que as combinações sorteadas são idênticas entre áreas para um mesmo algoritmo — o que só é verdade se todas as execuções partiram da mesma semente de amostragem. A semente de modelo é conferida na própria parametrização, que é fixa em `FIXOS`.

In [ ]:
arquivos = {(a, g): os.path.join(PASTA_SAIDA, f"{a}__{g}.csv")
            for a in AREAS for g in ESPACOS}
faltando = [k for k, f in arquivos.items() if not os.path.exists(f)]
assert not faltando, f"pares nao executados: {faltando}"

resultados = {k: pd.read_csv(f) for k, f in arquivos.items()}
for (a, g), r in resultados.items():
    assert len(r) == N_SORTEIOS + 1, f"{a}/{g}: orcamento incompleto ({len(r) - 1})"
    assert r["atual"].sum() == 1, f"{a}/{g}: configuracao atual ausente ou duplicada"

for g in ESPACOS:
    sorteados = {a: list(resultados[(a, g)].sort_values("ordem")
                         .loc[lambda d: ~d["atual"], "params"]) for a in AREAS}
    referencia = sorteados[AREAS[0]]
    assert all(v == referencia for v in sorteados.values()), (
        f"{g}: combinacoes sorteadas divergem entre areas")

print(f"{len(arquivos)} pares | {N_SORTEIOS} sorteios + configuracao atual em cada")
print(f"combinacoes sorteadas identicas entre areas para cada algoritmo "
      f"(semente de busca {SEMENTE_BUSCA}) | semente de modelo {SEMENTE_MODELO}")
print(f"total de ajustes avaliados: {len(arquivos) * (N_SORTEIOS + 1) * len(DOBRAS)}")

### 13. Tabela compartilhada

Consolida, por par de área e algoritmo, as métricas da configuração atual e da melhor combinação sorteada, o Δ de MAE contra a atual, o piso da área, a decisão e o motivo do empate quando houver. É esta tabela que vai para a planilha compartilhada e que alimenta o arquivo de hiperparâmetros do card de entrega.

In [ ]:
escolhas = pd.DataFrame([escolhe(r) for r in resultados.values()])
escolhas = escolhas.sort_values(["area", "MAE"]).reset_index(drop=True)
escolhas.to_csv(os.path.join(PASTA_SAIDA, "escolhas.csv"), index=False)

colunas = ["area", "algoritmo", "MAE_atual", "RMSE_atual", "R2_atual",
           "MAE_melhor", "RMSE_melhor", "R2_melhor", "delta_MAE", "piso",
           "delta_ep", "p_wilcoxon", "passa_significancia", "decisao",
           "motivo_empate", "sorteio_vencedor", "extremo"]
print(escolhas[colunas].to_string(index=False))

### 14. Leitura por área e por algoritmo

Duas visões da mesma tabela. A primeira mostra, por área, o melhor algoritmo depois da busca e o Δ que a busca produziu; a segunda mostra, por algoritmo, em quantas áreas a busca trocou a configuração.

In [ ]:
melhor_por_area = (escolhas.sort_values("MAE").groupby("area", as_index=False).first()
                   .sort_values("MAE")[["area", "algoritmo", "MAE", "MAE_atual",
                                        "delta_MAE", "piso", "decisao"]])
print("Melhor algoritmo por area, apos a busca:")
print(melhor_por_area.to_string(index=False))

resumo = (escolhas.assign(trocou=escolhas["decisao"].eq("melhor sorteio"))
          .groupby("algoritmo")
          .agg(areas_com_troca=("trocou", "sum"),
               delta_MAE_medio=("delta_MAE", "mean"),
               MAE_medio=("MAE", "mean"))
          .round(3).sort_values("MAE_medio"))
print("\nPor algoritmo, sobre as sete areas:")
print(resumo.to_string())

### 15. Extremos do suporte e saturação do orçamento

Duas verificações próprias da busca aleatória, apuradas sobre os 28 pares.

O primeiro bloco lista os pares cuja vencedora caiu nos 10% externos do suporte de alguma distribuição, indicação de que o suporte pode estar curto e de que o ótimo está fora dele. O segundo traça a curva do melhor MAE acumulado ao longo dos sorteios: se a vencedora aparece cedo e a curva fica plana até o fim, o orçamento saturou; se a melhora ocorre nos últimos sorteios, ele é curto e a busca deveria ser reaberta com orçamento maior.

In [ ]:
extremos = escolhas[escolhas["extremo"] != "-"][["area", "algoritmo", "extremo", "decisao"]]
print(f"Vencedoras no extremo do suporte: {len(extremos)} de {len(escolhas)} pares")
print(extremos.to_string(index=False) if len(extremos) else "nenhuma")

In [ ]:
marcos = [1, 5, 10, 15, 20, 25, N_SORTEIOS]
linhas = []
for (a, g), r in resultados.items():
    curva, indice = saturacao(r)
    linhas.append({"area": a, "algoritmo": g,
                   **{f"n={m}": round(curva[m - 1], 2) for m in marcos},
                   "sorteio_vencedor": indice})
curvas = pd.DataFrame(linhas).sort_values(["area", "algoritmo"])
curvas.to_csv(os.path.join(PASTA_SAIDA, "curvas_saturacao.csv"), index=False)
print("Melhor MAE acumulado (°C) por numero de sorteios:")
print(curvas.to_string(index=False))

tardias = curvas[curvas["sorteio_vencedor"] > N_SORTEIOS * 0.8]
print(f"\nPares cuja vencedora surgiu nos ultimos 20% do orcamento: {len(tardias)} de "
      f"{len(curvas)} | mediana do sorteio vencedor: "
      f"{int(curvas['sorteio_vencedor'].median())} de {N_SORTEIOS}")

## Parte IV - Entrega dos hiperparâmetros

Converte a tabela consolidada da Parte III no artefato lido pelos notebooks de mesclagem das Pessoas A e B. A entrega é um arquivo JSON, um módulo de leitura e a verificação de que o arquivo reproduz as métricas registradas.

Como as 28 comparações terminaram em empate pelo critério da seção 7, o artefato entrega, em todos os pares, a configuração atual do algoritmo. Isso não é ausência de resultado: a configuração entregue passou por 30 sorteios de busca e nenhuma alternativa a superou por margem que ultrapassasse o piso da área, o que é informação necessária às tarefas A1, A2, A3 e B1. Cada entrada carrega o campo `origem`, que distingue a configuração mantida da configuração vencedora da busca, para o caso de o critério ser revisto.

### 16. Construção da estrutura

O arquivo tem três blocos: `protocolo`, com a formulação sob a qual os hiperparâmetros valem; `busca`, com o método, o orçamento e as sementes; e `areas`, com uma entrada por par de área e algoritmo.

Os parâmetros são gravados completos, com os valores fixos (`n_estimators`, `random_state`, `n_jobs`) já embutidos, de modo que a mesclagem instancie o modelo sem precisar reconstruir nada. Os contínuos vão sem arredondamento: a serialização de `float` em JSON preserva o valor exato em ponto flutuante, e a seção 18 confere o ida e volta.

In [ ]:
from datetime import datetime, timezone

def raiz_projeto():
    '''Raiz do repositorio, ancorada no arquivo de particao e nao no diretorio do kernel.'''
    if PASTA_PROJETO:
        return os.path.abspath(PASTA_PROJETO)
    caminho = os.path.dirname(os.path.abspath(resolve("particao_treino_teste.csv")))
    if os.path.basename(caminho) in ("colabs_S3", "preparacao_testes", "notebooks"):
        caminho = os.path.dirname(caminho)      # dados em subpasta: sobe para a raiz
    return caminho


RAIZ = raiz_projeto()
PASTA_ARTEFATOS = os.path.join(RAIZ, "artefatos")
os.makedirs(PASTA_ARTEFATOS, exist_ok=True)
ARTEFATO = os.path.join(PASTA_ARTEFATOS, "hiperparametros.json")
print("Raiz do projeto:", RAIZ)

assert "escolhas" in dir(), "execute a Parte III antes desta celula"

areas_json = {}
for _, linha in escolhas.iterrows():
    alg = linha["algoritmo"]
    params = {**FIXOS[alg], **json.loads(linha["params_escolhidos"])}
    areas_json.setdefault(linha["area"], {})[alg] = {
        "params": params,
        "origem": ("melhor sorteio" if linha["decisao"] == "melhor sorteio"
                   else "configuracao atual"),
        "empate": bool(linha["decisao"] != "melhor sorteio"),
        "motivo_empate": linha["motivo_empate"],
        "MAE_validacao": float(linha["MAE"]),
        "RMSE_validacao": float(linha["RMSE"]),
        "R2_validacao": float(linha["R2"]),
        "MAE_atual": float(linha["MAE_atual"]),
        "MAE_melhor_sorteio": float(linha["MAE_melhor"]),
        "delta_MAE": float(linha["delta_MAE"]),
        "piso_area": float(linha["piso"]),
        "delta_ep": float(linha["delta_ep"]),
        "p_wilcoxon": float(linha["p_wilcoxon"]),
        "passa_significancia": bool(linha["passa_significancia"]),
        "sorteio_vencedor": linha["sorteio_vencedor"],
        "n_sorteios": N_SORTEIOS,
        "semente_busca": SEMENTE_BUSCA,
        "semente_modelo": SEMENTE_MODELO,
    }

artefato = {
    "gerado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "origem": "random_search.ipynb (Parte III)",
    "protocolo": {
        "versao_base": VERSAO,
        "tratamento_nulos": TRATAMENTO,
        "formulacao": "formato longo por area, com patamar e escala do componente "
                      "calculados dentro da dobra",
        "validacao": f"leave-one-group-out por configuracao, {len(DOBRAS)} dobras",
        "particao": "particao_treino_teste.csv",
        "criterio_troca": f"Wilcoxon pareado (p < {LIMIAR_P}) e ganho de MAE >= piso da area",
        "conjunto_teste": "nao utilizado",
    },
    "busca": {
        "metodo": "random search (ParameterSampler)",
        "n_sorteios": N_SORTEIOS,
        "semente_busca": SEMENTE_BUSCA,
        "semente_modelo": SEMENTE_MODELO,
        "algoritmos": sorted(ESPACOS),
        "areas": AREAS,
    },
    "areas": {a: areas_json[a] for a in AREAS},
}

with open(ARTEFATO, "w", encoding="utf-8") as f:
    json.dump(artefato, f, ensure_ascii=False, indent=2)

print("Artefato:", ARTEFATO)
print(f"{len(artefato['areas'])} areas x "
      f"{len(next(iter(artefato['areas'].values())))} algoritmos | "
      f"{os.path.getsize(ARTEFATO) / 1024:.1f} KB")

### 17. Módulo de leitura

As duas funções que a mesclagem usa são gravadas em `artefatos/hiperparametros.py`, ao lado do JSON, para que os notebooks das Pessoas A e B não precisem copiar código: `carrega` devolve o artefato inteiro e `modelo` devolve o estimador já instanciado. O Ridge sai como pipeline com escalonador, e não como regressor solto, porque o hiperparâmetro foi ajustado sob escalonamento — usá-lo sem o escalonador muda o significado do `alpha`.

O módulo depende apenas de `json`, `os` e `scikit-learn`.

In [ ]:
MODULO = os.path.join(PASTA_ARTEFATOS, "hiperparametros.py")
FONTE = '''"""Leitura dos hiperparametros ajustados por random search (Sprint 4).

Uso nos notebooks de mesclagem:

    import sys; sys.path.append("../artefatos")
    from hiperparametros import carrega, modelo

    arte = carrega()
    reg = modelo("coil_temps", "random_forest")      # estimador pronto para .fit
"""
import json
import os

from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PADRAO = os.path.join(os.path.dirname(os.path.abspath(__file__)), "hiperparametros.json")

CLASSES = {"random_forest": RandomForestRegressor,
           "extra_trees": ExtraTreesRegressor,
           "gradient_boosting": GradientBoostingRegressor}


def carrega(caminho=PADRAO):
    """Devolve o artefato inteiro."""
    with open(caminho, encoding="utf-8") as f:
        return json.load(f)


def params(area, algoritmo, caminho=PADRAO):
    """Devolve apenas os hiperparametros do par, ja com os valores fixos."""
    arte = carrega(caminho)
    if area not in arte["areas"]:
        raise KeyError(f"area desconhecida: {area}")
    if algoritmo not in arte["areas"][area]:
        raise KeyError(f"algoritmo desconhecido: {algoritmo}")
    return arte["areas"][area][algoritmo]["params"]


def modelo(area, algoritmo, caminho=PADRAO):
    """Instancia o estimador do par. Ridge sai como pipeline com escalonador."""
    p = params(area, algoritmo, caminho)
    if algoritmo == "ridge":
        return make_pipeline(StandardScaler(), Ridge(**p))
    return CLASSES[algoritmo](**p)
'''
with open(MODULO, "w", encoding="utf-8") as f:
    f.write(FONTE)

import importlib.util
spec = importlib.util.spec_from_file_location("hiperparametros", MODULO)
hp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hp)

print("Modulo:", MODULO)
print("exemplo:", hp.modelo("coil_temps", "random_forest"))
print("exemplo:", hp.modelo("oven_hmi", "ridge"))

### 18. Verificação da estrutura 

Quatro conferências, na ordem dos critérios de aceite.

A cobertura exige os quatro algoritmos nas sete áreas, sem lacuna, e todos os campos exigidos em cada entrada. A precisão confere que o valor lido do arquivo é bit a bit igual ao gravado, isto é, que a serialização não arredondou nada. A coerência confere que MAE, RMSE e R² do artefato coincidem com os da tabela compartilhada. A reprodução é a verificação forte: para os 28 pares, o modelo é instanciado a partir do arquivo, revalidado sob o mesmo protocolo, e o MAE resultante é comparado ao registrado.

In [ ]:
lido = hp.carrega(ARTEFATO)
CAMPOS = {"params", "origem", "empate", "MAE_validacao", "RMSE_validacao", "R2_validacao",
          "delta_MAE", "piso_area", "p_wilcoxon", "n_sorteios", "semente_busca",
          "semente_modelo", "sorteio_vencedor"}

faltando = [(a, g) for a in AREAS for g in ESPACOS if g not in lido["areas"].get(a, {})]
assert not faltando, f"lacunas no artefato: {faltando}"
for a in AREAS:
    for g in ESPACOS:
        ausentes = CAMPOS - set(lido["areas"][a][g])
        assert not ausentes, f"{a}/{g}: campos ausentes {ausentes}"
print(f"cobertura: {len(AREAS)} areas x {len(ESPACOS)} algoritmos, sem lacunas")

continuos = 0
for a in AREAS:
    for g in ESPACOS:
        for k, v in artefato["areas"][a][g]["params"].items():
            assert lido["areas"][a][g]["params"][k] == v, f"{a}/{g}/{k}: valor alterado"
            continuos += isinstance(v, float)
print(f"precisao: {continuos} parametros continuos preservados bit a bit no ida e volta")

conferencia = escolhas.set_index(["area", "algoritmo"])
for a in AREAS:
    for g in ESPACOS:
        e = lido["areas"][a][g]
        linha = conferencia.loc[(a, g)]
        for campo, coluna in (("MAE_validacao", "MAE"), ("RMSE_validacao", "RMSE"),
                              ("R2_validacao", "R2"), ("delta_MAE", "delta_MAE")):
            assert abs(e[campo] - linha[coluna]) < 1e-9, f"{a}/{g}/{campo}: divergente"
print("coerencia: metricas do artefato identicas as da tabela compartilhada")

In [ ]:
t0, linhas = time.perf_counter(), []
for a in AREAS:
    longo, X = carrega_area(a)
    for g in ESPACOS:
        registrado = lido["areas"][a][g]["MAE_validacao"]
        met, _ = avalia(longo, X, lambda: hp.modelo(a, g, ARTEFATO))
        linhas.append({"area": a, "algoritmo": g, "MAE_artefato": round(registrado, 3),
                       "MAE_reproduzido": round(met["MAE"], 3),
                       "diferenca": round(met["MAE"] - registrado, 6)})
reproducao = pd.DataFrame(linhas)
assert reproducao["diferenca"].abs().max() < 1e-3, "artefato nao reproduz o MAE registrado"
print(f"reproducao: {len(reproducao)} pares reinstanciados a partir do arquivo em "
      f"{(time.perf_counter() - t0) / 60:.1f} min | maior diferenca "
      f"{reproducao['diferenca'].abs().max():.2e} °C")
print(reproducao.to_string(index=False))

### 19. Resumo da entrega

Fecha os hiperparâmetros e onde está o arquivo com os mesmos, como lê-lo e o que cada entrada significa, com destaque para os pares em que a busca empatou e a configuração atual foi mantida.

In [ ]:
empates = [(a, g) for a in AREAS for g in ESPACOS if lido["areas"][a][g]["empate"]]
print(f"Artefato: {ARTEFATO}")
print(f"Modulo:   {MODULO}")
print(f"\nCobertura: {len(AREAS) * len(ESPACOS)} pares | "
      f"{len(empates)} empates, com a configuracao atual mantida | "
      f"{len(AREAS) * len(ESPACOS) - len(empates)} trocas pela busca")

if empates:
    print("\nPares em que o tuning empatou (campo origem = 'configuracao atual'):")
    resumo_empates = pd.DataFrame(
        [{"area": a, "algoritmo": g,
          "delta_MAE": lido["areas"][a][g]["delta_MAE"],
          "piso": lido["areas"][a][g]["piso_area"],
          "motivo": lido["areas"][a][g]["motivo_empate"],
          "passa_significancia": lido["areas"][a][g]["passa_significancia"]}
         for a, g in empates])
    print(resumo_empates.to_string(index=False))

print("\nUso na mesclagem:")
print('    import sys; sys.path.append("../artefatos")')
print("    from hiperparametros import carrega, modelo")
print('    reg = modelo("coil_temps", "random_forest")')